# Question 02 Scatterplot Planning (Scatterplot of Health vs. Education)

## Step 01: Load Data

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/clean/panel_1980_2010.csv")

print(df.shape)
print(df.columns.tolist())

(615, 19)
['Country Name', 'Country Code', 'Year', 'region', 'income', 'infant_mortality', 'life_expectancy', 'under5_mortality', 'literacy_rate', 'primary_enrollment', 'prog_to_secondary', 'secondary_enrollment', 'adolescent_fertility_per_1000_women', 'open_defecation_pct', 'sanitation_basic_pct', 'sanitation_safely_managed_pct', 'water_basic_pct', 'gdp_per_capita_PPP', 'gdp_per_capita_const2010usd']


## Step 02: Define Variable Groups (health vs. education; makes dropdowns easier to implement)

In [8]:
education_vars = [
    "literacy_rate",
    "primary_enrollment",
    "prog_to_secondary",    # not-very-good coverage
    "secondary_enrollment", # mostly-reliable coverage (1980-2020), but may have missing values for highly-developed countries
]

# below: all very good coverage 1980-2020
health_vars = [
    "life_expectancy",
    "infant_mortality",
    "under5_mortality",
]

## Step 03 (preliminary) - start with a single year for planning (we'll move to a dynamic year-slider later)

In [9]:
# 3.1. start with year = 2004
year = 2002

# 3.2. get data for all countries in only 2000 (as a stepping stone to eventual final product)
plot_df = df[df["Year"] == year].copy()

print(plot_df.shape)

(15, 19)


## Step 04 - first non-interactive scatterplot (preliminary baseline)

In [10]:
import plotly.express as px

# 4.1. Plot first scatterplot (of secondary enrollment versus under5 mortality)
fig = px.scatter(
    plot_df,
    x="secondary_enrollment",
    y="under5_mortality",
    color="income",
    hover_name="Country Name",
)

fig.show()

## Step 05-06 - Improving the Static Chart in Step 4 (post-baseline, pre-interaction)

Descriptive Tooltips, Trendline, Correlation comp+classify

1. Tooltips: (see country name, region, income level, and both x and y)
2. Trendline
3. Correlation (value and classification (weak positive, etc.))

*Note: temporarily: x="secondary_enrollment" and y="under5_mortality"*

In [11]:
# May need to run this install the first time
# %pip install statsmodels

import statsmodels.api as sm # for trendline

# 5.1. Plot first scatterplot (of secondary enrollment versus under5 mortality)
fig = px.scatter(
    plot_df,
    x="secondary_enrollment",   # TODO: make a user-selectable parameter
    y="under5_mortality",       # TODO: make a user-selectable parameter
    color="income",
    hover_name="Country Name",
    hover_data=[
        "region",
        "income",
        "life_expectancy",
        "under5_mortality"
    ],
    # trendline="ols"
)

# 5.3. Fit one regression line across all displayed countries
trend_df = plot_df[["secondary_enrollment", "under5_mortality"]].dropna()
x = trend_df["secondary_enrollment"]
y = trend_df["under5_mortality"]

# 5.4. create OLS regression
X_model = sm.add_constant(x)
model = sm.OLS(y, X_model).fit()
# print(model.summary())  # summarise model

# 5.5. Plot line: Generate x-range
x_line = np.linspace(
    x.min(),
    x.max(),
    100 # multiply decimals by 100 to get percents
)

# 5.6. Generate y-predictions
X_line = sm.add_constant(x_line)    # add bias (y-intercept)
y_pred = model.predict(X_line)

# 5.7. Add the all-country trendline (includes all income levels) to the scatterplot
fig.add_scatter(
    x=x_line,
    y=y_pred,
    mode="lines",
    name="Overall Trend",
    line=dict(
        color="gray",
        dash="dash",
        width=2,
    )
)

fig.show()

## Step 7-8. Compute Correlation via NumPy, then classify it (a signed scalar)

In [12]:
# 7. Compute correlation
corr_df = plot_df[  # select education and sanitation columns
    ["secondary_enrollment", "under5_mortality"]
].dropna()

correlation_value = corr_df.corr().iloc[0, 1]

print(f"Correlation = {correlation_value:.2f}")


# 8. Classify correlation
def classify_correlation(r):

    magnitude = abs(r)

    if magnitude < 0.2:
        return "Very Weak"

    elif magnitude < 0.4:
        return "Weak"

    elif magnitude < 0.6:
        return "Moderate"

    elif magnitude < 0.8:
        return "Strong"

    return "Very Strong"

# Usage of classify correlation example:
label = classify_correlation(correlation_value)
print(label)

Correlation = -0.87
Very Strong


# Steps 9-16 - Building-Blocks for Interactivity 

## Step 9. Year-Selection Slider Widget

In [ ]:
# 9.1. may have to run this install command at first
# %pip install ipywidgets

import ipywidgets as widgets

# 9.2. add year-slider widget
year_slider = widgets.IntSlider(
    value=2000,     # default position
    min=1980,
    max=2019,
    step=1,
    description="Year"
)

# 9.3. Dev-time inspect
year_slider

IntSlider(value=2000, description='Year', max=2019, min=1980)

## Step 10-11. Year-Selection Textbox + Sync selector and textbox

In [33]:
# 10. Year selection textbox
year_text = widgets.BoundedIntText(
    value=1990,
    min=1980,
    max=2020,
    description="Year"
)

# 11. Link them together. One update will update another. 
widgets.jslink(
    (year_slider, "value"),
    (year_text, "value")
)

# year_slider
year_text   # testing - type a year and press enter. The previous cell's slider should jump in sync. 

BoundedIntText(value=1990, description='Year', max=2020, min=1980)

## 12-13. X- and Y- axis dropdown

In [34]:
# 12. x-axis dropdown (e.g., education)
x_dropdown = widgets.Dropdown(
    options=education_vars,
    value="secondary_enrollment",
    description="X - edu"
)

x_dropdown

Dropdown(description='X - edu', index=3, options=('literacy_rate', 'primary_enrollment', 'prog_to_secondary', …

In [35]:
# 13. Y-axis dropdown (e.g., health outcome)
y_dropdown = widgets.Dropdown(
    options=health_vars,
    value="under5_mortality",
    description="Y - health"
)

y_dropdown

Dropdown(description='Y - health', index=2, options=('life_expectancy', 'infant_mortality', 'under5_mortality'…

## Step 14. Region Filter (E / SE Asia) - universal Q2-Q3

In [36]:
# Checkboxes for E / SE Asia toggle (interactive scatterplot)
    # by default - both selected (show all countries)
east_checkbox = widgets.Checkbox(
    value=True,
    description="East Asia"
)

southeast_checkbox = widgets.Checkbox(
    value=True,
    description="Southeast Asia"
)

## Step 15. Color Toggle (encode either region or income-level to color)

In [37]:
color_toggle = widgets.ToggleButtons(
    options=[
        "region",
        "income"
    ],
    description="Color Encoding Choice"
)

color_toggle

ToggleButtons(description='Color Encoding Choice', options=('region', 'income'), value='region')

## Step 16. Data Coverage Info Functions: What % of total countries have non-null data for this selection (year, x-metric, y-metric)

In [87]:
# 16.1. Reusable HTML Correlation Card
    # displays the strength and sign of the correlation between x-y
correlation_card = widgets.HTML(
    value="""
    <div style="
        background:#123b35;
        color:white;
        padding:15px;
        border-radius:10px;
        width:250px;
    ">
        <div style="font-size:12px;">
            CORRELATION
        </div>

        <div style="
            font-size:36px;
            font-weight:bold;
        ">
            --
        </div>

        <div>
            --
        </div>
    </div>
    """
)

In [88]:
# 16.2. coverage banner (shows what % of selected countries has non-null data for these two metrics)
coverage_banner = widgets.HTML(
    value=""
)

In [89]:
# (reusable) Helper function for coverage-details banner
#   "What % of total countries have non-null data for this selection (year, x-metric, y-metric)?"

# Returns: a 3-item tuple (% coverage, "_[desc]__ Coverage", "color_of_banner")

def coverage_details(data, variable):

    # 16.1. Total data 
    total = len(data)
    if total == 0:
        return (0, "No data", "danger")

    # 16.2. Non-null data
    available = data[variable].notna().sum()
    pct = available / total

    if pct >= 0.80:
        return (pct, "Good Coverage", "success")

    elif pct >= 0.40:
        return (pct, "Limited Coverage", "warning")

    else:
        return (pct, "Sparese Coverage", "danger")

## Step 17: Dynamic Scatterplot Function (this is the driver of the interactive Q2 scatterplot)

In [ ]:
def fit_regression_line_to_fig(fig, plot_df, x_var, y_var, x_min, x_max):
    # 5.3. Fit one regression line across all displayed countries
    trend_df = plot_df[[x_var, y_var]].dropna()
    x = trend_df[x_var]
    y = trend_df[y_var]

    # 5.4. create OLS regression
    X_model = sm.add_constant(x)
    model = sm.OLS(y, X_model).fit()
    # print(model.summary())  # summarise model

    # 5.5. Plot line: Generate x-range
    x_line = np.linspace(
        x_min,
        x_max,
        100 # multiply decimals by 100 to get percents
    )

    # 5.6. Generate y-predictions
    X_line = sm.add_constant(x_line)    # add bias (y-intercept)
    y_pred = model.predict(X_line)

    # 5.7. Add the all-country trendline (includes all income levels) to the scatterplot
    fig.add_scatter(
        x=x_line,
        y=y_pred,
        mode="lines",
        name="Overall Trend",
        line=dict(
            color="gray",
            dash="dash",
            width=2,
        )
    )

    return fig

In [96]:
# Compute global range per variable
# Purpose: so that the axes remain stable whilst the points move across time

axis_ranges = {}

for col in education_vars + health_vars:
    series = df[col].dropna()
    axis_ranges[col] = (
        series.min(),
        series.max()
    )

In [97]:
# axis ranges for different x/y variables
# Purpose: so that the axes remain stable whilst the points move across time

axis_ranges = {
    "literacy_rate": (50, 100),
    "primary_enrollment": (70, 120),
    "prog_to_secondary": (50, 120),
    "secondary_enrollment": (40, 120),
    "life_expectancy": (40, 90),
    "infant_mortality": (0, 150),
    "under5_mortality": (0, 150),
}

In [ ]:
def make_plot(
    year, 
    x_var,
    y_var,
    include_east,
    include_southeast,
    color_var,
):
    # 17.1. Include only selected regions (E/SE Asia)
    selected_regions = []
    if include_east:
        selected_regions.append("East Asia")
    if include_southeast:
        selected_regions.append("Southeast Asia")

    # 17.2. Filter data by year and selected region
    filtered_df = df[
        (df["Year"] == year)
        &
        (df["region"].isin(selected_regions))
    ].copy()

    # 17.3. PLOT the actual scatterplot
    fig = px.scatter(
        filtered_df,
        x = x_var,
        y = y_var,
        color = color_var,
        hover_name="Country Name",
        hover_data=[
            "region",
            "income",
            x_var,
            y_var,
        ],
        title=f"{x_var} vs {y_var} ({year})",
        # trendline="ols"
    )

    # 17.3. Fix the axes to a reasonable range, so that points move (instead of axes)
    xmin, xmax = axis_ranges[x_var]
    ymin, ymax = axis_ranges[y_var]
    padding_x = (xmax - xmin) * 0.05  # add 5% padding
    fig.update_xaxes(
        range=[xmin - padding_x, xmax + padding_x],
    )

    padding_y = (ymax - ymin) * 0.05
    fig.update_yaxes(
        range=[ymin - padding_y, ymax + padding_y]
    )

    # 17.4. Add improved (nice) labels to the axes
    label_map = {
        "literacy_rate": "Literacy Rate (%)",
        "primary_enrollment": "Primary Enrollment (%)",
        "prog_to_secondary": "Progression to Secondary (%)",
        "secondary_enrollment": "Secondary Enrollment (%)",
        "life_expectancy": "Life Expectancy (Years)",
        "infant_mortality": "Infant Mortality (per 1000)",
        "under5_mortality": "Under-5 Mortality (per 1000)",
    }
    fig.update_layout(
        xaxis_title=label_map[x_var],
        yaxis_title=label_map[y_var],
    )

    # 17.5. Fit a regression line (helper function above)
    fig = fit_regression_line_to_fig(fig, filtered_df, x_var, y_var, xmin, xmax)

    

    # 17.6. Calculate and display correlation
    corr_df = filtered_df[[x_var, y_var]].dropna()
    if len(corr_df) > 2:    # if correlation makes sense; 2 or more points
        correlation_value = corr_df.corr().iloc[0, 1]

        # Display correlation card
        relationship_desc = classify_correlation(
            correlation_value
        ) 
        correlation_card.value = f"""
        <div style="
            background:#123b35;
            color:white;
            padding:15px;
            border-radius:10px;
            width:250px;
        ">
            <div style="
                font-size:12px;
                letter-spacing:1px;
            ">
                CORRELATION
            </div>

            <div style="
                font-size:36px;
                font-weight:bold;
            ">
                {correlation_value:.2f}
            </div>

            <div>
                {relationship_desc}
            </div>
        </div>
        """




    # 17.7. Coverage banner (how many % of selected countries have non-null data for x and y?)
    x_pct, x_status, _ = coverage_details(
        filtered_df,
        x_var
    )
    y_pct, y_status, _ = coverage_details(
        filtered_df,
        y_var
    )
    # determine worst coverage (which of x and y is lower)
    coverage_pct = min(x_pct, y_pct)
    # build banner
    if coverage_pct >= 0.80:    # good coverage
        coverage_banner.value = f"""
        <div style="
            background:#d4edda;
            color:#155724;
            padding:10px;
            border-radius:6px;
        ">
            ✅ Good data coverage
            ({coverage_pct:.0%} countries reporting)
        </div>
        """
    elif coverage_pct >= 0.40:  # limited coverage
        coverage_banner.value = f"""
        <div style="
            background:#fff3cd;
            color:#856404;
            padding:10px;
            border-radius:6px;
        ">
            ⚠ Limited data coverage
            ({coverage_pct:.0%} countries reporting)
        </div>
        """
    else:
        coverage_banner.value = f"""
        <div style="
            background:#f8d7da;
            color:#721c24;
            padding:10px;
            border-radius:6px;
        ">
            ❌ Sparse data coverage
            ({coverage_pct:.0%} countries reporting)
        </div>
        """

    return fig



## Step 18: Plug (connect) the interactive widgets (Steps 9-15) into the dynamic scatterplot function (Step 17)

In [99]:
ui = widgets.interactive_output(
    make_plot,
    {
        "year": year_slider,
        "x_var": x_dropdown,
        "y_var": y_dropdown,
        "include_east": east_checkbox,
        "include_southeast": southeast_checkbox,
        "color_var": color_toggle,
    },
)

## Step 19: Display Dashboard

In [100]:
controls = widgets.VBox([
    year_slider,
    year_text,
    x_dropdown,
    y_dropdown,
    east_checkbox,
    southeast_checkbox,
    color_toggle,
])

dashboard = widgets.HBox([
    ui,
    widgets.VBox([
        correlation_card,
        coverage_banner,
        controls
    ])
])

dashboard